# Clean and standardize data in AbbVie AI Data Collection file

In [8]:
import pandas as pd
from datetime import datetime
import openpyxl
import re

In [2]:
input_file = 'AbbVie AI Data Collection.xlsx'
output_file = 'AbbVie_AI_Data_Collection_Cleaned.csv'

In [5]:
xls = pd.ExcelFile(input_file)
df_pr = pd.read_excel(xls, 'Press releases')
df_tw = pd.read_excel(xls, 'Twitter')

In [6]:
df_pr.head()

,title,date,verbatim,link,Radiotherapeutic,topics,audience,Notes
0,Allergan Aesthetics Announces Positive Topline...,2023-10-24,–All primary and secondary endpoints were met ...,https://news.abbvie.com/news/press-releases/al...,NaN,clinical treatment news/update,all,NaN
1,Allergan Aesthetics Supports Breast Cancer Pat...,2023-10-19,Ongoing Support for Breast Cancer Patients Thr...,https://news.abbvie.com/news/press-releases/al...,NaN,company announcement // other,all,NaN
2,AbbVie's SKYRIZI® (risankizumab) Versus STELAR...,2023-10-15,"SEQUENCE, a Phase 3 head-to-head study, compar...",https://news.abbvie.com/news/press-releases/ab...,NaN,company announcement // clinical trial,all,NaN
3,AbbVie Announces Upadacitinib (RINVOQ®) Met th...,2023-10-12,"At week 24, upadacitinib met the primary endpo...",https://news.abbvie.com/news/press-releases/ab...,NaN,company announcement // clinical trial,all,NaN
4,AbbVie Presents Long-Term Data Further Support...,2023-10-11,"Three ongoing, randomized, double-blind, multi...",https://news.abbvie.com/news/press-releases/ab...,NaN,clinical treatment news/update,all,NaN


In [7]:
df_pr['topics'].unique()

array(['clinical treatment news/update', 'company announcement // other',
       'company announcement // clinical trial',
       'company announcement // research & publications',
       'earnings call // financial results',
       'company announcement // conferences & outreach',
       'company announcement // motivation & support',
       'company announcement // financial results',
       'company announcement // patents',
       'partnership // research & publications',
       'company announcement // research & publication',
       'company announcement // conferences & outreach / research & publications',
       'company announcement // general disease info / awareness',
       'company announcement // new hires / employee update',
       'partnership // research & publication',
       'company announcement // patents / clinical treatment news/update',
       'company announcement // research & publications / motivation & support',
       'company announcement // conferencs & o

In [9]:
df_pr['topics'].nunique()

18

In [10]:
df_tw['topics'].nunique()

33

In [19]:
def clean_topic(topic):
    """Clean up the topic string by correcting common typos and standardizing formatting."""
    if pd.isna(topic):
        return topic
    
    topic = str(topic).strip().lower()

    topic = ' '.join(topic.split())

    # Fix typos
    topic = re.sub(r'^compant\b', 'company', topic)
    topic = re.sub(r'^company update\b', 'company announcement', topic)
    topic = re.sub(r'^partnershiphip\b', 'partnership', topic)
    topic = re.sub(r'^parnter\b', 'partnership', topic)
    topic = topic.replace('conferenes', 'conferences')
    topic = topic.replace('conference', 'conferences')
    topic = topic.replace('conferencs', 'conferences')

    topic = topic.replace('diease', 'disease')
    topic = topic.replace('diesase', 'disease')
    topic = topic.replace('othera', 'other')

    topic = topic.replace(' and ', ' & ')

    topic = re.sub(r'\bpublication\b(?!s)', 'publications', topic)
    topic = re.sub(r'\bpatent\b(?!s)', 'patents', topic)
    topic = re.sub(r'\bparent\b(?!s)', 'patents', topic)
    topic = re.sub(r'\bparents\b', 'patents', topic)
    
    topic = re.sub(r'\bhires\b', 'hire', topic)
    topic = re.sub(r'\bupdates\b', 'hire', topic)
    topic = re.sub(r'\bupate\b', 'hire', topic)

    topic = topic.replace('events', 'outreach')

    topic = re.sub(r' / awareness$', '', topic)

    return topic

In [23]:
def clean_topics_column(df, sheet_name):
    """Apply the clean_topic function to the 'topics' column of the DataFrame."""
    
    def process_row(topic_str):
        if pd.isna(topic_str):
            return topic_str
        
        # Split into individual topics
        topics = str(topic_str).split('//')
        
        # Clean each topic
        topics = [clean_topic(t) for t in topics]
        
        # Remove extra whitespace
        topics = [' '.join(t.split()) for t in topics]
        
        # Deduplicate while preserving order
        seen = set()
        cleaned_topics = []
        for t in topics:
            if t and t not in seen:
                seen.add(t)
                cleaned_topics.append(t)
        
        return cleaned_topics

    # Apply row-wise and assign back
    df['topics'] = df['topics'].apply(process_row)

    return df

In [24]:
df_pr_cleaned = clean_topics_column(df_pr, 'Press releases')
df_tw_cleaned = clean_topics_column(df_tw, 'Twitter')

In [27]:
df_pr_cleaned['topics'].head(100)

0                    [clinical treatment news/update]
1                       [company announcement, other]
2              [company announcement, clinical trial]
3              [company announcement, clinical trial]
4                    [clinical treatment news/update]
                           ...                       
95                    [company announcement, patents]
96                   [clinical treatment news/update]
97          [company announcement, financial results]
98    [company announcement, research & publications]
99                    [company announcement, patents]
Name: topics, Length: 100, dtype: object

In [15]:
df_tw_cleaned['topics'].nunique()

17

In [16]:
with pd.ExcelWriter('AbbVie_AI_Data_Collection_Cleaned.xlsx') as writer:
    df_pr_cleaned.to_excel(writer, sheet_name='Press releases', index=False)
    df_tw_cleaned.to_excel(writer, sheet_name='Twitter', index=False)